In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.fft import fft

print("TensorFlow:", tf.__version__)


TensorFlow: 2.19.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os, cv2, zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models


In [4]:
zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Dataset extracted")


Dataset extracted


In [5]:
import os

BASE_PATH = "/content/dataset/Parkinson's Dataset Final"

for root, dirs, files in os.walk(BASE_PATH):
    level = root.replace(BASE_PATH, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    for f in files[:5]:
        print(f"{subindent}{f}")
    if level >= 4:
        break


Parkinson's Dataset Final/
    New hand pd Dataset/
        Parkinson's/
            PatientSpiral/
                PatientSpiral/
                    sp2-P16.jpg
                    sp4-P24.jpg
                    sp3-P14.jpg
                    sp2-P29.jpg
                    sp3-P26.jpg


In [6]:
import pandas as pd
import os

BASE = "/content/dataset/Parkinson's Dataset Final"

records = []

def collect_images(label, root_dir):
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(".jpg"):
                records.append({
                    "image_path": os.path.join(root, f),
                    "label": label
                })

# Parkinson images
collect_images(
    1,
    os.path.join(BASE, "New hand pd Dataset", "Parkinson's")
)

# Healthy images
collect_images(
    0,
    os.path.join(BASE, "New hand pd Dataset", "Healthy")
)

df = pd.DataFrame(records)
print(df.head())
print("\nTotal samples:", len(df))
print("\nClass balance:\n", df["label"].value_counts())


                                          image_path  label
0  /content/dataset/Parkinson's Dataset Final/New...      1
1  /content/dataset/Parkinson's Dataset Final/New...      1
2  /content/dataset/Parkinson's Dataset Final/New...      1
3  /content/dataset/Parkinson's Dataset Final/New...      1
4  /content/dataset/Parkinson's Dataset Final/New...      1

Total samples: 512

Class balance:
 label
1    279
0    233
Name: count, dtype: int64


In [7]:
import re

signal_root = os.path.join(
    BASE,
    "New hand pd Dataset",
    "Parkinson's",
    "PatientSignal",
    "Signal"
)

signal_map = {}

for root, _, files in os.walk(signal_root):
    for f in files:
        if f.lower().endswith(".txt"):
            match = re.search(r'P\d+', f)
            if match:
                pid = match.group()
                signal_map[pid] = os.path.join(root, f)

print("Total signal files:", len(signal_map))
list(signal_map.items())[:5]


Total signal files: 31


[('P17',
  "/content/dataset/Parkinson's Dataset Final/New hand pd Dataset/Parkinson's/PatientSignal/Signal/sigSp1-P17.txt"),
 ('P21',
  "/content/dataset/Parkinson's Dataset Final/New hand pd Dataset/Parkinson's/PatientSignal/Signal/sigMea3-P21.txt"),
 ('P16',
  "/content/dataset/Parkinson's Dataset Final/New hand pd Dataset/Parkinson's/PatientSignal/Signal/sigSp1-P16.txt"),
 ('P9',
  "/content/dataset/Parkinson's Dataset Final/New hand pd Dataset/Parkinson's/PatientSignal/Signal/sigMea1-P9.txt"),
 ('P6',
  "/content/dataset/Parkinson's Dataset Final/New hand pd Dataset/Parkinson's/PatientSignal/Signal/sigMea4-P6.txt")]

In [8]:
import os
import pandas as pd

IMAGE_ROOT = "/content/dataset/Parkinson's Dataset Final/New hand pd Dataset"

records = []

def scan_images(base_dir, label):
    for root, _, files in os.walk(base_dir):
        for f in files:
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                records.append({
                    "path": os.path.join(root, f),
                    "label": label
                })

# Parkinson's images → label = 1
scan_images(
    os.path.join(IMAGE_ROOT, "Parkinson's"),
    label=1
)

# Healthy images → label = 0
scan_images(
    os.path.join(IMAGE_ROOT, "Healthy"),
    label=0
)

image_df = pd.DataFrame(records)

print("Total image samples:", len(image_df))
print("\nClass balance:")
print(image_df["label"].value_counts())


Total image samples: 512

Class balance:
label
1    279
0    233
Name: count, dtype: int64


In [9]:
import re

def extract_pid_from_image(path):
    match = re.search(r'P\d+', path)
    return match.group() if match else None

image_df["pid"] = image_df["path"].apply(extract_pid_from_image)

# signal_map must already exist (from your signal loading cell)
image_df_signal = image_df[image_df["pid"].isin(signal_map.keys())]

print("Images with signals:", len(image_df_signal))
print("\nClass balance (image + signal):")
print(image_df_signal["label"].value_counts())


Images with signals: 306

Class balance (image + signal):
label
1    279
0     27
Name: count, dtype: int64


In [10]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(image_df_signal["label"]),
    y=image_df_signal["label"]
)

class_weights = {
    0: class_weights[0],
    1: class_weights[1]
}

class_weights


{0: np.float64(5.666666666666667), 1: np.float64(0.5483870967741935)}

In [11]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    image_df_signal,
    test_size=0.2,
    stratify=image_df_signal["label"],
    random_state=42
)

print("Train:", train_df["label"].value_counts())
print("Test:", test_df["label"].value_counts())


Train: label
1    222
0     22
Name: count, dtype: int64
Test: label
1    57
0     5
Name: count, dtype: int64


In [12]:
import cv2

IMG_SIZE = 224

def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    return img[..., np.newaxis]


In [13]:
def load_signal_from_pid(pid):
    path = signal_map[pid]
    df = read_signal_file(path)
    return extract_signal_features(df)


In [14]:
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size=16, embed_dim=526):
        super().__init__()
        self.patch_size = patch_size
        self.proj = layers.Conv2D(
            filters=embed_dim,
            kernel_size=patch_size,
            strides=patch_size,
            padding="valid"
        )

    def call(self, x):
        x = self.proj(x)                    # (B, H/ps, W/ps, embed_dim)
        x = tf.reshape(x, (tf.shape(x)[0], -1, x.shape[-1]))
        return x


In [15]:
class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_embed = layers.Embedding(
            input_dim=num_patches,
            output_dim=embed_dim
        )

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[1], delta=1)
        return x + self.pos_embed(positions)


In [16]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.embed_dim = embed_dim

        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            output_shape=embed_dim
        )

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

        self.mlp = layers.Dense(embed_dim, activation="gelu")

        self.proj = None

    def build(self, input_shape):
        if input_shape[-1] != self.embed_dim:
            self.proj = layers.Dense(self.embed_dim)

    def call(self, x):
        shortcut = x
        if self.proj is not None:
            shortcut = self.proj(shortcut)

        attn_out = self.attn(x, x)
        x = self.norm1(shortcut + attn_out)

        mlp_out = self.mlp(x)
        x = self.norm2(x + mlp_out)

        return x


In [17]:
def build_custom_vit_extractor():
    inputs = tf.keras.Input(shape=(224, 224, 1))

    x = PatchEmbedding(patch_size=16, embed_dim=526)(inputs)
    x = PositionalEncoding(num_patches=196, embed_dim=526)(x)

    for dim in [526, 256, 128, 64, 32, 16, 8, 4]:
        x = TransformerBlock(embed_dim=dim)(x)

    x = layers.GlobalAveragePooling1D()(x)

    return tf.keras.Model(
        inputs,
        x,
        name="ViT_Image_Extractor"
    )


In [18]:
vit_extractor = build_custom_vit_extractor()
vit_extractor.summary()


Model: "ViT_Image_Extractor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding                 │ (None, None, 526)      │       135,182 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ (None, None, 526)      │       103,096 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, None, 526)      │     1,383,900 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, None, 256)      │       672,256 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, None, 128)      │       165,120 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, None, 64)       │        41,600 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ (None, None, 32)       │        10,560 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_5             │ (None, None, 16)       │         2,720 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_6             │ (None, None, 8)        │           720 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_7             │ (None, None, 4)        │           200 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 4)              │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,515,354 (9.60 MB)

 Trainable params: 2,515,354 (9.60 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
import pandas as pd
import numpy as np

def read_signal_file(path):
    """
    Reads Parkinson's pen signal text file.
    Automatically skips metadata lines.
    """
    data = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()

            # Skip metadata
            if not line or line.startswith("#") or "<" in line:
                continue

            parts = line.split()
            if len(parts) >= 2:
                try:
                    data.append([float(p) for p in parts])
                except:
                    pass

    df = pd.DataFrame(data)
    return df


In [25]:
from scipy.stats import skew, kurtosis
from scipy.fft import fft

def extract_signal_features(df):
    """
    Extracts robust statistical + frequency features
    from handwriting signal
    """

    features = []

    for col in df.columns:
        x = df[col].values

        if len(x) < 10:
            continue

        # Time-domain features
        features.extend([
            np.mean(x),
            np.std(x),
            np.min(x),
            np.max(x),
            np.median(x),
            skew(x),
            kurtosis(x)
        ])

        # Frequency-domain features
        fft_vals = np.abs(fft(x))
        features.extend([
            np.mean(fft_vals),
            np.std(fft_vals),
            np.max(fft_vals)
        ])

    return np.array(features, dtype=np.float32)


In [26]:
X_sig = []
for pid in image_df_signal["pid"]:
    X_sig.append(load_signal_from_pid(pid))

X_sig = np.array(X_sig)
print("Signal feature shape:", X_sig.shape)


Signal feature shape: (306, 60)


In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_sig = scaler.fit_transform(X_sig)


In [28]:
# One row per patient
patient_df = (
    image_df_signal
    .groupby("pid")
    .agg({"label": "first"})
    .reset_index()
)

print("Total patients:", len(patient_df))
print(patient_df["label"].value_counts())


Total patients: 31
label
1    31
Name: count, dtype: int64


In [29]:
from sklearn.model_selection import train_test_split

train_pids, test_pids = train_test_split(
    patient_df["pid"],
    test_size=0.2,
    stratify=patient_df["label"],
    random_state=42
)

train_pids = set(train_pids)
test_pids  = set(test_pids)

print("Train patients:", len(train_pids))
print("Test patients :", len(test_pids))


Train patients: 24
Test patients : 7


In [30]:
train_df = image_df_signal[image_df_signal["pid"].isin(train_pids)]
test_df  = image_df_signal[image_df_signal["pid"].isin(test_pids)]

print("Train images:", len(train_df))
print("Test images :", len(test_df))

print("\nTrain label balance:")
print(train_df["label"].value_counts())

print("\nTest label balance:")
print(test_df["label"].value_counts())


Train images: 238
Test images : 68

Train label balance:
label
1    216
0     22
Name: count, dtype: int64

Test label balance:
label
1    63
0     5
Name: count, dtype: int64


In [31]:
X_img_train = np.array([load_image(p) for p in train_df["path"]])
X_img_test  = np.array([load_image(p) for p in test_df["path"]])

y_train = train_df["label"].values
y_test  = test_df["label"].values


In [32]:
X_sig_train = np.array([
    load_signal_from_pid(pid) for pid in train_df["pid"]
])

X_sig_test = np.array([
    load_signal_from_pid(pid) for pid in test_df["pid"]
])

print("Image train:", X_img_train.shape)
print("Signal train:", X_sig_train.shape)


Image train: (238, 224, 224, 1)
Signal train: (238, 60)


In [33]:
from sklearn.preprocessing import StandardScaler

sig_scaler = StandardScaler()
X_sig_train = sig_scaler.fit_transform(X_sig_train)
X_sig_test  = sig_scaler.transform(X_sig_test)


In [34]:
def focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        eps = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1. - eps)

        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        return -alpha * tf.pow(1 - pt, gamma) * tf.math.log(pt)
    return loss


In [35]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=3e-4,
    clipnorm=1.0
)


In [37]:
X_sig = []
for pid in image_df_signal["pid"]:
    X_sig.append(load_signal_from_pid(pid))

X_sig = np.array(X_sig)

X_sig_train, X_sig_test = train_test_split(
    X_sig,
    test_df["label"],
    test_size=0.2,
    stratify=test_df["label"],
    random_state=42
)


ValueError: Found input variables with inconsistent numbers of samples: [306, 68]